# Tutorial 01: Silicon Electronic Structure

In this tutorial you will run the complete PAOFLOW post-processing workflow on bulk silicon: build a PAO Hamiltonian from atomic projections, interpolate the electronic band structure, compute the density of states, and evaluate Boltzmann transport tensors.

**What you will learn**

- How Kohn–Sham wavefunctions are projected onto pseudo-atomic orbital (PAO) bases
- How a real-space PAO Hamiltonian $H(\mathbf{R})$ is built by Fourier transform
- How interpolation produces smooth bands on arbitrarily dense k-grids
- How the density of states and transport tensors are computed from interpolated eigenvalues


## Required Input

> This notebook assumes that a Quantum ESPRESSO `.save` directory is already available. Users may generate it themselves or download the tutorial assets from the [PAOFLOW Releases page](https://github.com/marcobn/PAOFLOW/releases).

The `.save` directory must contain:

- `data-file-schema.xml` — crystal structure, k-point grid, and Kohn–Sham eigenvalues
- `atomic_proj.xml` — projection amplitudes and overlap matrices

The overlap matrix is required and is written by `projwfc.x` only when `lwrite_overlaps = .true.` is set in the input.


In [ ]:
# Adjust these paths if needed
savedir = 'silicon.save'
outputdir = 'output'

## 1. Initialise PAOFLOW

`PAOFLOW.PAOFLOW(...)` reads `data-file-schema.xml` to load the crystal structure, k-point grid, and Kohn–Sham eigenvalues, then prepares all internal data structures. All output files are written to `outputdir`.


In [ ]:
from PAOFLOW import PAOFLOW

paoflow = PAOFLOW.PAOFLOW(
    savedir=savedir,
    outputdir=outputdir,
    smearing='gauss',
    npool=1,
    verbose=True,
)

## 2. Read Atomic Projections

PAOFLOW represents the electronic structure in a **pseudo-atomic orbital (PAO)** basis — localised functions $|\phi_\alpha\rangle$ derived from the isolated-atom Kohn–Sham solutions for each angular momentum channel in the pseudopotential. For silicon ($3s^2\,3p^2$ valence configuration), this gives one $s$ and three $p$ orbitals per atom: eight PAOs for the two-atom unit cell.

The projection amplitudes

$$A_{n\alpha}(\mathbf{k}) = \langle\phi_\alpha|\psi_{n\mathbf{k}}\rangle$$

quantify how much of Kohn–Sham band $n$ at k-point $\mathbf{k}$ is captured by PAO $\alpha$. The overlap matrix $S_{\alpha\beta}(\mathbf{k})$ between PAOs on different sites is needed for the subsequent Löwdin orthogonalisation.


In [ ]:
paoflow.read_atomic_proj_QE()

## 3. Check Projectability

The **projectability** of band $n$ at k-point $\mathbf{k}$,

$$p_{n\mathbf{k}} = \sum_\alpha |A_{n\alpha}(\mathbf{k})|^2,$$

measures how completely the PAO basis captures that band. Values near 1 indicate a good representation; values well below 1 correspond to high-energy, plane-wave-like states not captured by localised orbitals.

Bands with $p_{n\mathbf{k}}$ below the threshold (default 0.95) are excluded from Hamiltonian construction. For silicon, the four valence bands and four lowest conduction bands are well within the PAO window.


In [ ]:
paoflow.projectability()

## 4. Build the PAO Hamiltonian

With the projection amplitudes in hand, the k-space PAO Hamiltonian is assembled as

$$\tilde{H}_{\alpha\beta}(\mathbf{k}) = \sum_n A^*_{n\alpha}(\mathbf{k})\,\varepsilon_{n\mathbf{k}}\,A_{n\beta}(\mathbf{k})$$

after Löwdin orthogonalisation of the PAO basis using $S(\mathbf{k})$. A discrete Fourier transform then maps this to real space:

$$H_{\alpha\beta}(\mathbf{R}) = \frac{1}{N_k}\sum_{\mathbf{k}} e^{-i\mathbf{k}\cdot\mathbf{R}}\,\tilde{H}_{\alpha\beta}(\mathbf{k}).$$

The real-space Hamiltonian $H(\mathbf{R})$ decays rapidly with distance — only a handful of Wigner–Seitz shells carry significant weight. This compact, first-principles tight-binding model is the basis for all subsequent interpolation: evaluating bands at any new k-point costs only a fast Fourier sum over these shells followed by an $8\times8$ matrix diagonalisation.

**Output:** `HRs.npy`, `R.npy`


In [ ]:
paoflow.pao_hamiltonian()

## 5. Compute the Band Structure

`bands()` evaluates $\tilde{H}(\mathbf{k}) = \sum_{\mathbf{R}} e^{i\mathbf{k}\cdot\mathbf{R}}\,H(\mathbf{R})$ along the standard FCC high-symmetry path ($\Gamma$–$X$–$W$–$K$–$\Gamma$–$L$–$U$–$W$–$L$–$K$, selected by `ibrav=2`) at 2000 k-points and diagonalises the result.

The silicon band structure displays the characteristic **indirect gap**: the valence band maximum sits at $\Gamma$ while the conduction band minimum lies between $\Gamma$ and $X$. PBE-GGA underestimates the gap (~0.6 eV calculated vs 1.1 eV experimental) — a well-known limitation of the approximate exchange-correlation functional.

**Output:** `bands_0.dat` (two columns: k-path coordinate, energy in eV, one block per band)


In [ ]:
paoflow.bands(ibrav=2, nk=2000)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

bands_file = Path(outputdir) / 'bands_0.dat'

if not bands_file.exists():
    print(f'Missing {bands_file}. Run the bands cell above first.')
else:
    with open(bands_file) as f:
        content = f.read()

    # Split into per-band blocks (separated by blank lines)
    data = []
    current = []
    for line in content.splitlines():
        if line.strip():
            current.append([float(x) for x in line.split()])
        else:
            if current:
                data.append(np.array(current))
                current = []
    if current:
        data.append(np.array(current))

    fig, ax = plt.subplots(figsize=(6, 5))
    for band in data:
        ax.plot(band[:, 0], band[:, 1], 'k-', lw=0.8)
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax.set_xlabel('k-path')
    ax.set_ylabel('Energy (eV)')
    ax.set_title('Silicon — interpolated band structure')
    plt.tight_layout()
    plt.show()

## 6. Interpolate the Brillouin-Zone Grid

`interpolated_hamiltonian()` doubles the k-grid density by Fourier interpolation (e.g. 12×12×12 → 24×24×24), providing a finer BZ sampling for accurate DOS and transport integrations.

`pao_eigh()` diagonalises $\tilde{H}(\mathbf{k})$ at every k-point on the interpolated grid, storing the eigenvalues and eigenvectors needed downstream.


In [ ]:
paoflow.interpolated_hamiltonian()
paoflow.pao_eigh()

## 7. Momentum Matrix Elements and Adaptive Smearing

`gradient_and_momenta()` computes the k-gradient of the Hamiltonian,

$$\nabla_{\mathbf{k}} H(\mathbf{k}) = i\sum_{\mathbf{R}} \mathbf{R}\,H(\mathbf{R})\,e^{i\mathbf{k}\cdot\mathbf{R}},$$

and from it the momentum matrix elements $\mathbf{p}_{nm}(\mathbf{k})$ and group velocities $\mathbf{v}_{n\mathbf{k}} = \hbar^{-1}\nabla_{\mathbf{k}}\varepsilon_{n\mathbf{k}}$ needed for transport.

`adaptive_smearing()` assigns each (band, k-point) a Gaussian broadening width proportional to $|\nabla_{\mathbf{k}}\varepsilon_{n\mathbf{k}}|$. Flat bands receive more broadening than dispersive ones, improving BZ integration accuracy without over-smearing sharp features.


In [ ]:
paoflow.gradient_and_momenta()
paoflow.adaptive_smearing()

## 8. Compute and Plot the Density of States

The electronic density of states,

$$g(\varepsilon) = \frac{1}{N_k}\sum_{n,\mathbf{k}}\delta(\varepsilon - \varepsilon_{n\mathbf{k}}),$$

is evaluated with adaptive Gaussian broadening at 1000 energy points spanning the full valence band and the lower conduction edge.

The silicon DOS shows a clear gap separating the valence and conduction manifolds. Van Hove singularities at band edges appear as steps or peaks. The total integrated weight under the valence DOS equals the number of valence electrons per cell (8 for Si).

**Output:** `dosdk_0.dat` (two columns: energy in eV, states/eV/cell)


In [ ]:
paoflow.dos(emin=-12.0, emax=2.2, ne=1000)

In [ ]:
dos_file = Path(outputdir) / 'dosdk_0.dat'

if not dos_file.exists():
    print(f'Missing {dos_file}. Run the DOS cell above first.')
else:
    dos_data = np.loadtxt(dos_file)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(dos_data[:, 0], dos_data[:, 1], 'b-', lw=1.2)
    ax.axvline(0, color='gray', ls='--', lw=0.5)
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('DOS (states/eV/cell)')
    ax.set_title('Silicon — density of states')
    plt.tight_layout()
    plt.show()

## 9. Compute and Plot Transport Tensors

Within the Boltzmann transport framework and the constant relaxation-time approximation, the electrical conductivity tensor is

$$\sigma_{\mu\nu}(E) = e^2\tau\,\frac{1}{N_k}\sum_{n,\mathbf{k}} v^\mu_{n\mathbf{k}}\,v^\nu_{n\mathbf{k}} \left(-\frac{\partial f}{\partial\varepsilon}\right)_{\varepsilon_{n\mathbf{k}}},$$

where $\tau$ is the (assumed constant) relaxation time and $f$ is the Fermi–Dirac distribution. PAOFLOW also computes the Seebeck coefficient and electronic thermal conductivity from energy-weighted moments of the same expression. All tensors are functions of chemical potential.

**Output:** `sigmagauss_0.dat`, `Seebeckgauss_0.dat`, `kappagauss_0.dat`


In [ ]:
paoflow.transport(emin=-12.0, emax=2.2)
paoflow.finish_execution()

In [ ]:
sigma_file = Path(outputdir) / 'sigmagauss_0.dat'

if not sigma_file.exists():
    print(f'Missing {sigma_file}. Run the transport cell above first.')
else:
    sigma_data = np.loadtxt(sigma_file)
    if sigma_data.ndim == 1:
        sigma_data = sigma_data.reshape(1, -1)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(sigma_data[:, 0], sigma_data[:, 1], lw=1.2)
    ax.axvline(0, color='gray', ls='--', lw=0.5)
    ax.set_xlabel('Chemical potential (eV)')
    ax.set_ylabel('Conductivity per relaxation time')
    ax.set_title('Silicon — electrical conductivity')
    plt.tight_layout()
    plt.show()